Embedding using DistilBert and fine tuning using DistilBert



In [11]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
data = pd.read_csv("training_data_lowercase.csv",sep='\t', names=['label', 'title'])
print(data.shape)
data.fillna("",inplace=True)
print(data.head())


(34152, 2)
   label                                              title
0      0  donald trump sends out embarrassing new year‚s...
1      0  drunk bragging trump staffer started russian c...
2      0  sheriff david clarke becomes an internet joke ...
3      0  trump is so obsessed he even has obama‚s name ...
4      0  pope francis just called out donald trump duri...


as part of pre proc we are able to see color codes in csv which are incorrectly interpretted by vs code
they are not color code but simply corresponds to episode num

we also see video/picture are there in some data points, while these are just metadata it could change the meaning of the sentence once we remove punctuations

we also see that this metadata in enclosed in () in Training sample while its enclosed in [] in testing data, so we might need different pre processing

In [13]:
from preProc import normalize_text

data["clean_text"] = data["title"].apply(normalize_text)
print(data.head)

<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  
0      donald trump sends out embarrassing new year s...  
1      drunk bragging trump staffer started rus

In [14]:
from sklearn.model_selection import train_test_split
X=data.drop(columns=['label'])
y=data['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_val.shape}")
print(f"Training output size: {y_train.shape}")
print(f"Testing output size: {y_val.shape}")

Training set size: (27321, 2)
Testing set size: (6831, 2)
Training output size: (27321,)
Testing output size: (6831,)


DistilBERT Embeddings

In [31]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score
import numpy as np

# 1. Load Tokenizer and Model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
# num_labels=2 for binary classification (Real vs Fake)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Convert your DataFrame to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(X_train[["clean_text"]].assign(label=y_train))
val_dataset = Dataset.from_pandas(X_val[["clean_text"]].assign(label=y_val))

print(train_dataset[0])

# 3. Tokenization Function
def tokenize_function(examples):
    return tokenizer(examples["clean_text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

print(tokenized_train[0])

# 4. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,              # 3 epochs is usually enough for fine-tuning
    eval_strategy='epoch',      # Evaluate after each epoch
    learning_rate=2e-5,              # Crucial: use a very small learning rate for fine-tuning
    weight_decay=0.01,
)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}
# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)




Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'clean_text': 'so', 'label': 0, '__index_level_0__': 8891}


Map:   0%|          | 0/27321 [00:00<?, ? examples/s]

Map:   0%|          | 0/6831 [00:00<?, ? examples/s]

{'clean_text': 'so', 'label': 0, '__index_level_0__': 8891, 'input_ids': [101, 2061, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [32]:
# 6. Start Fine-Tuning
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.144923,0.102920,0.970575
2,0.060526,0.128027,0.975113
3,0.020910,0.145298,0.977602


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10248, training_loss=0.09240401298342786, metrics={'train_runtime': 1124.6661, 'train_samples_per_second': 72.878, 'train_steps_per_second': 9.112, 'total_flos': 2714356349010432.0, 'train_loss': 0.09240401298342786, 'epoch': 3.0})